In [ ]:
import lightgmb

In [ ]:
import lightgbm

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.competition_download('iaa-ai-assisted-hackathon-2026-part-1')

print("Path to competition files:", path)

In [ ]:
pip install kagglehub

In [ ]:
import pandas as pd
from pathlib import Path

DATA = Path("../data")
tables = {f.stem: pd.read_csv(f) for f in DATA.glob("*.csv")}

for name, df in tables.items():
    print(f"\n=== {name}: {df.shape} ===")
    print(df.dtypes.to_string())
    print("missing:", df.isna().sum()[df.isna().sum() > 0].to_dict())

In [ ]:
import os
from pathlib import Path

print("현재 위치:", os.getcwd())
print("data 폴더 존재:", Path("../data").exists())
for p in Path("../data").rglob("*"):
    print(p)

In [ ]:
import os
from pathlib import Path

print("현재 위치:", os.getcwd())
print("data 폴더 존재:", Path("../data").exists())
for p in Path("../data").rglob("*"):
    print(p)

In [ ]:
import pandas as pd
from pathlib import Path

DATA = Path("../data")
tables = {f.stem: pd.read_parquet(f) for f in sorted(DATA.glob("*.parquet"))}

for name, df in tables.items():
    print(f"\n=== {name}: {df.shape} ===")
    print(df.dtypes.to_string())
    na = df.isna().sum()
    print("missing:", na[na > 0].to_dict())

In [ ]:
%pip install pyarrow

In [ ]:
import os
from pathlib import Path

print("현재 위치:", os.getcwd())
print("data 폴더 존재:", Path("../data").exists())
for p in Path("../data").rglob("*"):
    print(p)

In [ ]:
import pandas as pd
from pathlib import Path

DATA = Path("../data")
tables = {f.stem: pd.read_parquet(f) for f in sorted(DATA.glob("*.parquet"))}

for name, df in tables.items():
    print(f"\n=== {name}: {df.shape} ===")
    print(df.dtypes.to_string())
    na = df.isna().sum()
    print("missing:", na[na > 0].to_dict())

In [ ]:
for name in ["claims", "policies_motor"]:
    df = tables[name]
    print(f"\n=== {name}: {df.shape} ===")
    print(df.dtypes.to_string())
    na = df.isna().sum()
    print("missing:", na[na > 0].to_dict())
    print(df.head(3).T)

In [ ]:
claims = tables["claims"].copy()
claims["loss_date"] = pd.to_datetime(claims["loss_date"])
claims["report_date"] = pd.to_datetime(claims["report_date"])
claims["report_lag"] = (claims["report_date"] - claims["loss_date"]).dt.days
claims["loss_year"] = claims["loss_date"].dt.year

print(claims["status"].value_counts(), "\n")
print(pd.crosstab(claims["line"], claims["peril"]), "\n")
print(pd.crosstab(claims["loss_year"], claims["status"]), "\n")
print(claims[["paid_to_date", "incurred_to_date", "recoveries", "report_lag"]].describe().T, "\n")

# policies 컬럼 전체, 키 유일성, train/test 컬럼 차이
pm, tm = tables["policies_motor"], tables["test_motor"]
print("policies_motor 컬럼:", list(pm.columns))
print("policy_id 유일?", pm["policy_id"].is_unique if "policy_id" in pm else "policy_id 없음")
print("train에만 있는 컬럼:", set(pm.columns) - set(tm.columns))
print("test에만 있는 컬럼:", set(tm.columns) - set(pm.columns))

In [ ]:
pm = tables["policies_motor"].copy()
pm["period_start"] = pd.to_datetime(pm["period_start"])
pm["period_end"] = pd.to_datetime(pm["period_end"])

# 1) 컬럼 전체 (excess 찾기)
print(list(pm.columns), "\n")

# 2) 클레임 → 행 매칭: loss_date가 기간 안에 들어가는지
cm = claims[claims["line"] == "motor"]
m = cm.merge(pm[["row_id", "policy_id", "period_start", "period_end"]], on="policy_id", how="left")
m["in_period"] = (m["loss_date"] >= m["period_start"]) & (m["loss_date"] <= m["period_end"])
matched = m[m["in_period"]]
print("motor 클레임 수:", cm.shape[0])
print("기간 안에 매칭된 클레임 수:", matched["claim_id"].nunique())
print("두 행 이상에 매칭된 클레임:", (matched.groupby("claim_id").size() > 1).sum())
print("어느 기간에도 안 맞는 클레임:", cm.shape[0] - matched["claim_id"].nunique(), "\n")

# 3) repudiated인데 금액이 있는지
rep = claims[claims["status"] == "repudiated"]
print(rep[["paid_to_date", "incurred_to_date", "recoveries"]].describe().T, "\n")

# 4) 보고 지연 분포 (IBNR 판단용)
print(claims.groupby("line")["report_lag"].describe(percentiles=[.5, .9, .99]))

In [ ]:
# 1) excess 관련 컬럼 찾기 (아까 목록이 잘림)
for t in ["policies_motor", "policies_buildings", "policies_contents"]:
    print(t, [c for c in tables[t].columns])
    print()

# 2) 매칭 안 된 297건은 왜?
unm_ids = set(cm["claim_id"]) - set(matched["claim_id"])
unm = m[m["claim_id"].isin(unm_ids)].drop_duplicates("claim_id")
print("loss_year 분포:\n", unm["loss_date"].dt.year.value_counts().sort_index())
print("policy_id가 policies에 아예 없는 건:", unm["period_start"].isna().sum(), "\n")

# 3) motor peril별 보고 지연과 open 비율
cmo = claims[claims["line"] == "motor"]
print(cmo.groupby("peril")["report_lag"].describe(percentiles=[.5, .9, .99])[["count", "50%", "90%", "99%", "max"]])
print()
print(pd.crosstab(cmo["peril"], cmo["status"], normalize="index").round(3))

In [ ]:
# 1) 컬럼을 한 줄씩
for t in ["policies_motor", "policies_buildings", "policies_contents"]:
    print(f"\n--- {t} ---")
    for c in tables[t].columns:
        print(" ", c)

# 2) bodily_injury 사고연도별 건수와 open 비율
bi = claims[(claims["line"] == "motor") & (claims["peril"] == "bodily_injury")]
print(pd.crosstab(bi["loss_year"], bi["status"], margins=True))

In [ ]:
for t in ["policies_motor", "policies_buildings", "policies_contents"]:
    cols = list(tables[t].columns)
    print(t, len(cols), "개 컬럼")
    print("  excess 후보:", [c for c in cols if any(k in c.lower() for k in ["excess", "deduct", "sum_insured", "limit"])])
    print("  overnight_parking 이후:", cols[cols.index("overnight_parking")+1:] if "overnight_parking" in cols else cols[-12:])

In [ ]:
pm = tables["policies_motor"].copy()
pm["period_start"] = pd.to_datetime(pm["period_start"])
pm["period_end"] = pd.to_datetime(pm["period_end"])
pm["total_excess"] = pm["voluntary_excess"] + pm["compulsory_excess"]

cm = claims[(claims["line"] == "motor") & (claims["status"] != "repudiated")]
m = cm.merge(pm[["row_id", "policy_id", "period_start", "period_end", "total_excess"]], on="policy_id")
m = m[(m["loss_date"] >= m["period_start"]) & (m["loss_date"] <= m["period_end"])]

closed = m[m["status"] == "closed"].copy()
closed["below_excess"] = closed["incurred_to_date"] <= closed["total_excess"]
closed["ratio"] = closed["incurred_to_date"] / closed["total_excess"]

print(pm["total_excess"].describe(), "\n")
print(closed.groupby("peril").agg(
    n=("claim_id", "size"),
    pct_below_excess=("below_excess", "mean"),
    incurred_min=("incurred_to_date", "min"),
    incurred_p10=("incurred_to_date", lambda x: x.quantile(.1)),
    incurred_median=("incurred_to_date", "median"),
    excess_median=("total_excess", "median"),
).round(3))
print("\nincurred가 0인 closed 클레임 비율:", (closed["incurred_to_date"] == 0).mean().round(3))

In [ ]:
m = m.copy()
m["gross_minus_excess"] = (m["incurred_to_date"] - m["total_excess"]).clip(lower=0)
m["net_A"] = (m["gross_minus_excess"] - m["recoveries"]).clip(lower=0)                              # excess 먼저
m["net_B"] = (m["incurred_to_date"] - m["recoveries"] - m["total_excess"]).clip(lower=0)            # recovery 먼저

print("recoveries > 0 비율:", (m["recoveries"] > 0).mean().round(3))
print("recoveries > incurred 인 건:", (m["recoveries"] > m["incurred_to_date"]).sum())
print("net_A 합계:", m["net_A"].sum().round(0), "/ net_B 합계:", m["net_B"].sum().round(0))
print(m.groupby("peril")[["incurred_to_date", "recoveries", "net_A", "net_B"]].sum().round(0))

# 계약 기간(row_id)별 타깃
target = m.groupby("row_id")["net_A"].sum().rename("target")
pm_t = pm.merge(target, on="row_id", how="left").fillna({"target": 0})
pm_t["year"] = pm_t["period_start"].dt.year
print("\n연도별 노출당 비용:")
print(pm_t.groupby("year").apply(lambda d: d["target"].sum() / d["exposure"].sum()).round(1))

In [ ]:
def build_target(line):
    pol = tables[f"policies_{line}"].copy()
    pol["period_start"] = pd.to_datetime(pol["period_start"])
    pol["period_end"] = pd.to_datetime(pol["period_end"])
    pol["total_excess"] = pol["voluntary_excess"] + pol["compulsory_excess"]

    c = claims[(claims["line"] == line) & (claims["status"] != "repudiated")]
    mm = c.merge(pol[["row_id", "policy_id", "period_start", "period_end", "total_excess"]], on="policy_id")
    mm = mm[(mm["loss_date"] >= mm["period_start"]) & (mm["loss_date"] <= mm["period_end"])].copy()

    n_total = c["claim_id"].nunique()
    n_matched = mm["claim_id"].nunique()
    n_dup = (mm.groupby("claim_id").size() > 1).sum()

    mm["net"] = ((mm["incurred_to_date"] - mm["total_excess"]).clip(lower=0) - mm["recoveries"]).clip(lower=0)

    tgt = mm.groupby("row_id")["net"].sum().rename("target")
    pol = pol.merge(tgt, on="row_id", how="left").fillna({"target": 0})
    pol["year"] = pol["period_start"].dt.year

    print(f"\n===== {line} =====")
    print(f"클레임 {n_total} / 매칭 {n_matched} ({n_matched/n_total:.1%}) / 중복 매칭 {n_dup}")
    print("peril별 비용 비중:\n", (mm.groupby("peril")["net"].sum() / mm["net"].sum()).round(3).sort_values(ascending=False))
    print("연도별 노출당 비용:\n", pol.groupby("year").apply(lambda d: d["target"].sum() / d["exposure"].sum()).round(1))
    print("open 비율 (사고연도별):\n", pd.crosstab(mm["loss_date"].dt.year, mm["status"], normalize="index")["open"].round(3))
    return pol, mm

results = {line: build_target(line) for line in ["motor", "buildings", "contents"]}

In [ ]:
for line in ["buildings", "contents"]:
    pol, mm = results[line]
    print(f"\n===== {line} =====")
    print("peril별 비중:", (mm.groupby("peril")["net"].sum() / mm["net"].sum()).round(3).sort_values(ascending=False).to_dict())
    print("노출당 비용:", pol.groupby("year").apply(lambda d: d["target"].sum() / d["exposure"].sum()).round(1).to_dict())
    print("open 비율:", pd.crosstab(mm["loss_date"].dt.year, mm["status"], normalize="index")["open"].round(3).to_dict())

In [ ]:
### Target build — all lines
- **Check:** Refactored target logic into `build_target(line)`; motor output matches the manual calculation exactly.
- **Finding (buildings):** Net cost per exposure rises each year — 2021: 268 / 2022: 292 / 2023: 305 / 2024: 325 — despite 25% of 2024 claims still open. Suggests an upward trend of roughly 6–7% p.a. Flood (29%) and fire (24%) dominate cost.
- **Finding (contents):** 2021: 64 / 2022: 66 / 2023: 73 / 2024: 73. Moderate upward trend; 2024 likely understated (13% open). Theft (43%) and flood (28%) dominate.
- **Implication:** Motor's main issue is development (bodily injury); buildings' main issue is trend projection to 2025 and separating weather effects from inflation.
- **Next step:** Check whether `weather` includes 2025 values for test addresses.

In [ ]:
### Target build — all lines
- **Check:** Refactored target logic into `build_target(line)`; motor output matches the manual calculation exactly.
- **Finding (buildings):** Net cost per exposure rises each year — 2021: 268 / 2022: 292 / 2023: 305 / 2024: 325 — despite 25% of 2024 claims still open. Suggests an upward trend of roughly 6–7% p.a. Flood (29%) and fire (24%) dominate cost.
- **Finding (contents):** 2021: 64 / 2022: 66 / 2023: 73 / 2024: 73. Moderate upward trend; 2024 likely understated (13% open). Theft (43%) and flood (28%) dominate.
- **Implication:** Motor's main issue is development (bodily injury); buildings' main issue is trend projection to 2025 and separating weather effects from inflation.
- **Next step:** Check whether `weather` includes 2025 values for test addresses.

In [ ]:
import pyarrow.parquet as pq

f = pq.ParquetFile("claims.parquet")
print(f.schema_arrow)                      # 컬럼 타입
print(f.metadata.metadata)                 # 파일 수준 메타데이터 (설명이 있으면 여기에)
for field in f.schema_arrow:
    if field.metadata:
        print(field.name, field.metadata)  # 컬럼별 설명

In [ ]:
import pyarrow.parquet as pq
from pathlib import Path

files = sorted(Path.home().joinpath("IAA").rglob("*.parquet"))
files = [p for p in files if ".venv" not in p.parts]
print([str(p) for p in files])

for p in files:
    f = pq.ParquetFile(p)
    print(f"\n===== {p.name} ({f.metadata.num_rows:,} rows) =====")
    print(f.schema_arrow)
    meta = f.metadata.metadata or {}
    for k, v in meta.items():
        k = k.decode()
        if k != "ARROW:schema":          # 내부용 스키마는 건너뜀
            print("[file meta]", k, v.decode()[:500])
    for field in f.schema_arrow:
        if field.metadata:
            print("[col meta]", field.name, field.metadata)

In [ ]:
import pandas as pd
D = "/Users/woojinji/IAA/data/"
claims = pd.read_parquet(D + "claims.parquet")
motor  = pd.read_parquet(D + "policies_motor.parquet")
tmotor = pd.read_parquet(D + "test_motor.parquet")

# 1. 상태값별 건수와 금액 (repudiated 외에 0원 상태가 있는지)
print(claims.groupby(["line", "status"])[["paid_to_date", "incurred_to_date", "recoveries"]]
      .agg(["count", "max"]))

# 2. test 기간 (예측 대상 연도)
for name, df in [("train", motor), ("test", tmotor)]:
    print(name, df.period_start.min(), df.period_start.max(), df.period_end.max())

# 3. test policy_id가 train에 있는지 (이력 피처 가능 여부)
print("test policies seen in train:", tmotor.policy_id.isin(motor.policy_id).mean())

# 4. 기간 경계: 앞 기간 끝과 다음 기간 시작의 간격
m = motor.sort_values(["policy_id", "period_start"])
gap = (m.groupby("policy_id").period_start.shift(-1) - m.period_end).dt.days
print(gap.value_counts().head())

# 5. excess가 peril마다 적용되는지 단서: closed 클레임 중 incurred <= excess 비율
mc = claims[(claims.line == "motor") & (claims.status == "closed")].merge(
    motor[["policy_id", "period_start", "period_end", "voluntary_excess", "compulsory_excess"]],
    on="policy_id")
mc = mc[(mc.loss_date >= mc.period_start) & (mc.loss_date <= mc.period_end)]
mc["excess"] = mc.voluntary_excess + mc.compulsory_excess
print(mc.groupby("peril").apply(lambda g: (g.incurred_to_date <= g.excess).mean()))

In [ ]:
for df in (claims, motor, tmotor):
    for c in df.columns:
        if c.endswith("_date") or c in ("period_start", "period_end"):
            df[c] = pd.to_datetime(df[c])

In [ ]:
import pandas as pd

D = "/Users/woojinji/IAA/data/"
claims = pd.read_parquet(D + "claims.parquet")
motor  = pd.read_parquet(D + "policies_motor.parquet")
tmotor = pd.read_parquet(D + "test_motor.parquet")

# 날짜 컬럼을 datetime으로 변환
for df in (claims, motor, tmotor):
    for c in df.columns:
        if c.endswith("_date") or c in ("period_start", "period_end"):
            df[c] = pd.to_datetime(df[c])

# 1. 상태값별 건수와 금액 (repudiated 외에 0원 상태가 있는지)
print("\n[1] status by line")
print(claims.groupby(["line", "status"])[["paid_to_date", "incurred_to_date", "recoveries"]]
      .agg(["count", "max"]))

# 2. train/test 기간 (예측 대상 연도)
print("\n[2] period ranges")
for name, df in [("train", motor), ("test", tmotor)]:
    print(name, df.period_start.min().date(), "~", df.period_start.max().date(),
          "| last period_end:", df.period_end.max().date())
print("last loss_date:", claims.loss_date.max().date(),
      "| last report_date:", claims.report_date.max().date())

# 3. test policy_id가 train에 있는지 (이력 피처 가능 여부)
print("\n[3] test policies seen in train:",
      round(tmotor.policy_id.isin(motor.policy_id).mean(), 4))

# 4. 기간 경계: 앞 기간 끝과 다음 기간 시작의 간격(일)
print("\n[4] gap between period_end and next period_start (days)")
m = motor.sort_values(["policy_id", "period_start"])
gap = (m.groupby("policy_id").period_start.shift(-1) - m.period_end).dt.days
print(gap.value_counts().head(10))

# 5. excess 적용 단서: closed 클레임 중 incurred <= excess 비율 (peril별)
print("\n[5] share of closed motor claims with incurred <= excess, by peril")
mc = claims[(claims.line == "motor") & (claims.status == "closed")].merge(
    motor[["policy_id", "period_start", "period_end",
           "voluntary_excess", "compulsory_excess"]],
    on="policy_id")
mc = mc[(mc.loss_date >= mc.period_start) & (mc.loss_date <= mc.period_end)]
mc["excess"] = mc.voluntary_excess + mc.compulsory_excess
print(mc.groupby("peril").apply(lambda g: (g.incurred_to_date <= g.excess).mean()).round(3))

In [ ]:
import pandas as pd

D = "/Users/woojinji/IAA/data/"
claims = pd.read_parquet(D + "claims.parquet")
motor  = pd.read_parquet(D + "policies_motor.parquet")

# 날짜 컬럼만 정확히 지정해서 변환
for c in ["loss_date", "report_date"]:
    claims[c] = pd.to_datetime(claims[c])
for c in ["period_start", "period_end"]:
    motor[c] = pd.to_datetime(motor[c])

# 1. 상태값별 금액
print("\n[1] status by line")
print(claims.groupby(["line", "status"])[["paid_to_date", "incurred_to_date", "recoveries"]]
      .agg(["count", "sum", "max"]).round(0))

# 5. closed motor 클레임 중 incurred <= excess 비율 (peril별)
print("\n[5] share of closed motor claims with incurred <= excess, by peril")
mc = claims[(claims.line == "motor") & (claims.status == "closed")].merge(
    motor[["policy_id", "period_start", "period_end",
           "voluntary_excess", "compulsory_excess"]],
    on="policy_id")
mc = mc[(mc.loss_date >= mc.period_start) & (mc.loss_date <= mc.period_end)]
mc["excess"] = mc.voluntary_excess + mc.compulsory_excess
summary = mc.groupby("peril").agg(
    n=("claim_id", "size"),
    share_le_excess=("incurred_to_date", lambda s: (s <= mc.loc[s.index, "excess"]).mean()),
    median_incurred=("incurred_to_date", "median"),
    median_excess=("excess", "median"),
)
print(summary.round(3))

# 덤: exposure 분포 (기간이 연말에서 잘리는지)
print("\n[6] exposure distribution")
print(motor.exposure.describe().round(3))

In [ ]:
import numpy as np

# A. exposure 이상값: 날짜로 계산한 기간과 비교
for name, df in [("train", motor), ("test", pd.read_parquet(D + "test_motor.parquet"))]:
    df = df.copy()
    df["days"] = (pd.to_datetime(df.period_end) - pd.to_datetime(df.period_start)).dt.days + 1
    df["exp_from_dates"] = df.days / 365
    bad = df.exposure > 1.01
    print(f"\n[A] {name}: exposure > 1.01 → {bad.sum():,} rows ({bad.mean():.2%})")
    print(df.loc[bad, ["exposure", "days", "exp_from_dates"]].head(10))
    print("exposure == days for bad rows:", (df.loc[bad, "exposure"] == df.loc[bad, "days"]).mean())
    if name == "train":
        print(df.loc[bad].groupby(df.period_start.dt.year).size())

# B. 11.9m 클레임 확인
print("\n[B] largest motor claims")
print(claims[claims.line == "motor"].nlargest(5, "incurred_to_date")
      [["claim_id", "peril", "status", "loss_date", "report_date",
        "paid_to_date", "incurred_to_date"]])

# C. 배상책임 peril에 excess 적용 여부에 따른 motor 순비용 차이
m2 = claims[(claims.line == "motor") & (claims.status != "repudiated")].merge(
    motor[["policy_id", "period_start", "period_end",
           "voluntary_excess", "compulsory_excess"]], on="policy_id")
m2 = m2[(m2.loss_date >= m2.period_start) & (m2.loss_date <= m2.period_end)]
m2["excess"] = m2.voluntary_excess + m2.compulsory_excess
liab = m2.peril.isin(["bodily_injury", "third_party_prop"])
net_all  = np.maximum(np.maximum(m2.incurred_to_date - m2.excess, 0) - m2.recoveries, 0)
net_liab = np.where(liab, np.maximum(m2.incurred_to_date - m2.recoveries, 0), net_all)
print("\n[C] motor net cost by peril (m)")
print(pd.DataFrame({"excess_all": net_all, "no_excess_on_liability": net_liab, "peril": m2.peril})
      .groupby("peril").sum().div(1e6).round(2))
print("total:", round(net_all.sum() / 1e6, 1), "vs", round(net_liab.sum() / 1e6, 1))

In [ ]:
import numpy as np
import pandas as pd

D = "/Users/woojinji/IAA/data/"
LINES = ["motor", "buildings", "contents"]

def load_policy(fname):
    df = pd.read_parquet(D + fname)
    df["period_start"] = pd.to_datetime(df.period_start)
    df["period_end"] = pd.to_datetime(df.period_end)
    df["days"] = (df.period_end - df.period_start).dt.days + 1
    df["exp_dates"] = df.days / 365
    return df

pol  = {l: load_policy(f"policies_{l}.parquet") for l in LINES}
test = {l: load_policy(f"test_{l}.parquet") for l in LINES}

def classify_exposure(df):
    ok       = np.isclose(df.exposure, df.exp_dates, atol=1e-3)
    basis360 = np.isclose(df.exposure, df.days / 360, atol=5e-4) & ~ok
    in_days  = np.isclose(df.exposure, df.days) & ~ok
    other    = ~(ok | basis360 | in_days)
    return pd.Series(np.select([ok, basis360, in_days], ["ok", "basis_360", "in_days"], "other"),
                     index=df.index)

for l in LINES:
    for name, df in [("train", pol[l]), ("test", test[l])]:
        df["exp_flag"] = classify_exposure(df)
        print(f"\n[{name} {l}]")
        print(df.exp_flag.value_counts())

    # 정상 행은 원래 값, 나머지는 날짜 기준 값으로 교체
    df = pol[l]
    df["exposure_clean"] = np.where(df.exp_flag == "ok", df.exposure, df.exp_dates)
    test[l]["exposure_clean"] = test[l].exposure

    by_year = df.groupby(df.period_start.dt.year)[["exposure", "exposure_clean"]].sum()
    by_year["change_%"] = (by_year.exposure_clean / by_year.exposure - 1) * 100
    print(by_year.round(1))

In [ ]:
claims = pd.read_parquet(D + "claims.parquet")
for c in ["loss_date", "report_date"]:
    claims[c] = pd.to_datetime(claims[c])
claims = claims[claims.status != "repudiated"]

LIAB = {"bodily_injury", "third_party_prop"}

def match_claims(line, excess_on_liability=True):
    """클레임을 계약 기간에 연결하고 클레임별 순비용 계산"""
    p = pol[line][["row_id", "policy_id", "period_start", "period_end",
                   "voluntary_excess", "compulsory_excess"]]
    c = claims[claims.line == line].merge(p, on="policy_id")
    c = c[(c.loss_date >= c.period_start) & (c.loss_date <= c.period_end)].copy()
    c["excess"] = c.voluntary_excess + c.compulsory_excess
    if not excess_on_liability:
        c.loc[c.peril.isin(LIAB), "excess"] = 0
    c["net"] = np.maximum(np.maximum(c.incurred_to_date - c.excess, 0) - c.recoveries, 0)
    return c

def build_target(line, excess_on_liability=True):
    """row_id별 타깃 (클레임 없는 행은 0)"""
    c = match_claims(line, excess_on_liability)
    t = c.groupby("row_id").net.sum()
    return pol[line].row_id.map(t).fillna(0).values

def cost_per_exposure(line, target):
    df = pol[line]
    g = pd.DataFrame({"year": df.period_start.dt.year, "net": target,
                      "exp": df.exposure_clean}).groupby("year").sum()
    return (g.net / g.exp).round(1)

out = {"motor_excess_all": cost_per_exposure("motor", build_target("motor", True)),
       "motor_no_excess_liab": cost_per_exposure("motor", build_target("motor", False)),
       "buildings": cost_per_exposure("buildings", build_target("buildings")),
       "contents": cost_per_exposure("contents", build_target("contents"))}
print(pd.DataFrame(out))

In [ ]:
bi = match_claims("motor", excess_on_liability=False)
bi = bi[bi.peril == "bodily_injury"]

print("[BI incurred percentiles]")
print(bi.incurred_to_date.quantile([0.5, 0.9, 0.99, 0.999]).round(0))

total = bi.net.sum()
for cap in [250_000, 500_000, 1_000_000, 2_000_000]:
    excess_part = np.maximum(bi.net - cap, 0)
    print(f"cap {cap:>9,}: {(bi.net > cap).sum():>3} claims, "
          f"{excess_part.sum() / total:.1%} of BI cost above cap")

print("\n[BI cost above 1m by loss year]")
bi["year"] = bi.loss_date.dt.year
print(bi.assign(above=np.maximum(bi.net - 1_000_000, 0))
        .groupby("year")[["net", "above"]].sum().div(1e6).round(2))

In [ ]:
# 1. 같은 계약·같은 사고일에 클레임이 여러 개인 경우 (motor)
mc_all = match_claims("motor", excess_on_liability=False)
multi = mc_all.groupby(["row_id", "loss_date"]).size()
print("[1] claims per (row_id, loss_date):")
print(multi.value_counts().sort_index())
events = mc_all.set_index(["row_id", "loss_date"]).loc[multi[multi > 1].index]
print("perils in multi-claim events:")
print(events.groupby(level=[0, 1]).peril.apply(lambda s: " + ".join(sorted(s))).value_counts().head(10))

# 2. 시작 연도와 종료 연도가 다른 행
print("\n[2] rows spanning two calendar years:")
for l in LINES:
    df = pol[l]
    print(l, (df.period_start.dt.year != df.period_end.dt.year).sum())

# 3. buildings·contents 검산: 클레임이 2건 이상인 행 2개씩 직접 확인
print("\n[3] manual check")
for l in ["buildings", "contents"]:
    c = match_claims(l)
    target = pd.Series(build_target(l), index=pol[l].row_id)
    for rid in c.row_id.value_counts().loc[lambda s: s >= 2].index[:2]:
        rows = c[c.row_id == rid][["peril", "incurred_to_date", "excess", "recoveries", "net"]]
        manual = sum(max(max(r.incurred_to_date - r.excess, 0) - r.recoveries, 0)
                     for r in rows.itertuples())
        print(f"\n{l} {rid}")
        print(rows.to_string(index=False))
        print(f"manual = {manual:,.2f} | build_target = {target[rid]:,.2f}")

In [ ]:
raw = pd.read_parquet(D + "claims.parquet")
for c in ["loss_date", "report_date"]:
    raw[c] = pd.to_datetime(raw[c])

# 1. claim_id 자체가 중복인 경우
print("[1] duplicated claim_id:", raw.claim_id.duplicated().sum())

# 2. claim_id만 다르고 나머지가 전부 같은 경우
key_full = ["policy_id", "line", "peril", "loss_date", "report_date", "status",
            "paid_to_date", "incurred_to_date", "recoveries"]
dup_full = raw.duplicated(subset=key_full, keep="first")

# 3. 금액은 같은데 보고일이나 상태가 다른 경우 (애매한 경우)
key_amt = ["policy_id", "line", "peril", "loss_date", "incurred_to_date"]
dup_amt = raw.duplicated(subset=key_amt, keep="first") & ~dup_full

print("\n[2] exact duplicates except claim_id, by line")
print(raw[dup_full].groupby("line").agg(n=("claim_id", "size"),
                                        incurred_m=("incurred_to_date", lambda s: s.sum() / 1e6)).round(2))
print("\n[3] same amount, different report_date/status, by line")
print(raw[dup_amt].groupby("line").size())

# 중복이 연도별로 고르게 퍼져 있는지 (심어 둔 함정인지 판단)
print("\n[4] exact duplicates by loss year")
print(raw[dup_full].groupby([raw.line, raw.loss_date.dt.year]).size().unstack(fill_value=0))

# 중복 쌍 예시: 다른 컬럼(repairer_id, third_party_id)도 같은지
ex = raw[raw.duplicated(subset=key_full, keep=False)].sort_values(key_full).head(6)
print("\n[5] example pairs")
print(ex[["claim_id", "policy_id", "peril", "loss_date", "report_date",
          "incurred_to_date", "repairer_id", "third_party_id"]].to_string(index=False))

In [ ]:
# 정리된 클레임: 중복 제거 + repudiated 제외
dup_key = ["policy_id", "line", "peril", "loss_date", "report_date", "status",
           "paid_to_date", "incurred_to_date", "recoveries"]
claims = raw[~raw.duplicated(subset=dup_key, keep="first")]
claims = claims[claims.status != "repudiated"].copy()
print("claims after cleaning:", len(claims))

# 노출당 비용 다시 계산 (exposure 수정본, motor는 배상책임 excess 미적용 기본안)
new = pd.DataFrame({
    "motor": cost_per_exposure("motor", build_target("motor", excess_on_liability=False)),
    "buildings": cost_per_exposure("buildings", build_target("buildings")),
    "contents": cost_per_exposure("contents", build_target("contents")),
})
print(new)

In [ ]:
VAL = pd.Timestamp("2024-12-30")   # 데이터 기준일 (마지막 report_date)
CAP = 500_000

mc = match_claims("motor", excess_on_liability=False)
mc["net_cap"] = np.minimum(mc.net, CAP)
mc["lag"] = (mc.report_date - mc.loss_date).dt.days
mc["ly"] = mc.loss_date.dt.year

def completeness(peril, year):
    """2021년 사고의 보고 지연 분포로, 해당 사고연도가 기준일까지 보고됐을 비율"""
    lags = np.sort(mc.loc[(mc.peril == peril) & (mc.ly == 2021), "lag"].values)
    days = pd.date_range(f"{year}-01-01", f"{year}-12-31")
    ages = (VAL - days).days.values
    return (np.searchsorted(lags, ages, side="right") / len(lags)).mean()

exp_by_year = pol["motor"].groupby(pol["motor"].period_start.dt.year).exposure_clean.sum()

rows = []
for peril in mc.peril.unique():
    for y in [2021, 2022, 2023, 2024]:
        reported = mc.loc[(mc.peril == peril) & (mc.ly == y), "net_cap"].sum()
        c = completeness(peril, y)
        rows.append({"peril": peril, "year": y, "reported_m": reported / 1e6,
                     "completeness": c, "ultimate_m": reported / c / 1e6})
dev = pd.DataFrame(rows)
print(dev.round(3).to_string(index=False))

# 연도별 motor 노출당 비용: 보고 기준 vs 보정 후 (cap 적용, 하중 제외)
summary = dev.groupby("year")[["reported_m", "ultimate_m"]].sum()
summary["reported_per_exp"] = summary.reported_m * 1e6 / exp_by_year
summary["ultimate_per_exp"] = summary.ultimate_m * 1e6 / exp_by_year
print(summary.round(1))

# 대형 사고 하중: cap 초과분을 4년 전체 노출로 나눔
loading = np.maximum(mc.net - CAP, 0).sum() / exp_by_year.sum()
print(f"\nlarge-loss loading per exposure: {loading:.1f}")

In [ ]:
bi = mc[mc.peril == "bodily_injury"].copy()

# 1. 2021년 BI: 보고 지연 구간별 평균 금액
ref = bi[bi.ly == 2021].copy()
ref["lag_bucket"] = pd.cut(ref.lag, [-1, 30, 90, 180, 365, 730, 5000],
                           labels=["~1m", "1-3m", "3-6m", "6-12m", "1-2y", "2y+"])
print("[1] 2021 BI: average cost by report lag")
print(ref.groupby("lag_bucket", observed=True).agg(n=("claim_id", "size"),
                                                   avg_net=("net_cap", "mean")).round(0))

# 2. 금액 가중 완성도 vs 건수 완성도
def completeness_bi(year, by_amount):
    r = ref.sort_values("lag")
    w = r.net_cap.values if by_amount else np.ones(len(r))
    cum = np.cumsum(w) / w.sum()
    ages = (VAL - pd.date_range(f"{year}-01-01", f"{year}-12-31")).days.values
    idx = np.searchsorted(r.lag.values, ages, side="right")
    return np.where(idx > 0, cum[idx - 1], 0).mean()

rows = []
for y in [2021, 2022, 2023, 2024]:
    reported = bi.loc[bi.ly == y, "net_cap"].sum()
    c_n, c_a = completeness_bi(y, False), completeness_bi(y, True)
    rows.append({"year": y, "reported_m": reported / 1e6,
                 "compl_count": c_n, "compl_amount": c_a,
                 "ult_count_m": reported / c_n / 1e6, "ult_amount_m": reported / c_a / 1e6})
print("\n[2] BI completeness: count-based vs amount-based")
print(pd.DataFrame(rows).round(3).to_string(index=False))

# 3. 사고연도별 빈도와 심도 (open/closed 구분)
g = bi.groupby("ly").agg(n=("claim_id", "size"), net_m=("net_cap", lambda s: s.sum() / 1e6),
                         open_share=("status", lambda s: (s == "open").mean()))
g["compl"] = [completeness_bi(y, False) for y in g.index]
g["freq_per_1000"] = g.n / g.compl / exp_by_year * 1000
g["avg_sev"] = g.net_m * 1e6 / g.n
print("\n[3] BI frequency and severity by loss year")
print(g.round(3))
print("\navg severity by status")
print(bi.groupby(["ly", "status"]).net_cap.mean().unstack().round(0))

In [ ]:
mc["lq"] = mc.loss_date.dt.to_period("Q")

# 1. BI 분기별 심도 (평균, 중앙값)
print("[1] BI severity by loss quarter")
q = mc[mc.peril == "bodily_injury"].groupby("lq").net_cap.agg(["size", "mean", "median"]).round(0)
print(q.to_string())

# 2. 보고 지연 1년 이내 건만으로 연도별 비교
print("\n[2] BI severity, claims reported within 365 days only")
same = mc[(mc.peril == "bodily_injury") & (mc.lag <= 365)]
print(same.groupby("ly").net_cap.agg(["size", "mean", "median"]).round(0))

# 3. peril별 연도 심도 (중앙값) 비교
print("\n[3] median severity by peril and loss year")
print(mc.groupby(["peril", "ly"]).net_cap.median().unstack().round(0))

In [ ]:
rows = []
for peril in mc.peril.unique():
    for y in [2021, 2022, 2023, 2024]:
        s = mc[(mc.peril == peril) & (mc.ly == y)]
        c = completeness(peril, y)
        rows.append({"peril": peril, "year": y,
                     "freq_per_1000": len(s) / c / exp_by_year[y] * 1000,
                     "sev_gross": np.minimum(s.incurred_to_date, CAP).mean()})
fs = pd.DataFrame(rows)

freq = fs.pivot(index="peril", columns="year", values="freq_per_1000")
sev = fs.pivot(index="peril", columns="year", values="sev_gross")

def annual_change(t):
    out = (t[2024] / t[2021]) ** (1 / 3) - 1
    out["bodily_injury"] = t.loc["bodily_injury", 2024] / t.loc["bodily_injury", 2023] - 1
    return out

freq["annual_%"] = annual_change(freq) * 100
sev["annual_%"] = annual_change(sev) * 100
print("[frequency per 1,000 exposures]")
print(freq.round(2))
print("\n[gross severity, capped at 500k]")
print(sev.round(0))